# Gridless PCB Router — Colab Training Notebook

Trains `DualStreamRouter` with PPO on curriculum PCB boards, with live plots and board snapshots each epoch, and checkpoints saved to **your Google Drive** so training survives Colab disconnects.

Run the cells in order, top to bottom. If Colab disconnects or you close the tab, just reopen this notebook and re-run all cells — training resumes from the last checkpoint automatically (see "Resuming after a disconnect" near the bottom).

Background: [PROJECT.md](https://github.com/Klutzhehe/Router-v2/blob/main/PROJECT.md) has the full architecture writeup; [docs/reward-function.md](https://github.com/Klutzhehe/Router-v2/blob/main/docs/reward-function.md) has the reward math.

## Step 1 — Mount Drive and get the code

Clones the repo into your Drive the first time; every run after that just `git pull`s so you get the latest code without re-cloning.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

REPO_URL = "https://github.com/Klutzhehe/Router-v2.git"
REPO_DIR = "/content/drive/MyDrive/Router-v2"
CKPT_DIR = "/content/drive/MyDrive/Router-v2-checkpoints"

import os
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} "{REPO_DIR}"
else:
    !cd "{REPO_DIR}" && git pull

os.makedirs(CKPT_DIR, exist_ok=True)
print("repo:", REPO_DIR)
print("checkpoints:", CKPT_DIR)

In [ ]:
import sys
sys.path.insert(0, f"{REPO_DIR}/python")

import json
import time
from collections import deque
from pathlib import Path

import numpy as np
import torch

from pcb_router.config import EnvConfig
from pcb_router.env import RoutingEnv, VecRoutingEnv
from pcb_router.generator import STAGES, generate_board
from pcb_router.model import DualStreamRouter
from pcb_router.ppo import (PPO, PPOConfig, collect_rollout, collect_rollout_vec,
                            load_checkpoint, save_checkpoint, to_torch)
from pcb_router.viz import TrainingMonitor, show_board_inline

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
# Net count is RNG-dependent now (real component placement, not fixed random
# pads) -- print the component mix and layer/stack-up info instead of a
# static net count.
for i, s in enumerate(STAGES):
    print(f"  stage {i}: {s.components}, {s.layers} layers, {s.size}x{s.size}mm board, "
          f"{s.n_diff_pairs} diff pairs")

## Step 2 — Configuration

`RUN_NAME` names the checkpoint (`{RUN_NAME}.pt`) and log file (`{RUN_NAME}_history.jsonl`) in `CKPT_DIR`. Use a new `RUN_NAME` to start a fresh, independent run; reuse one to resume it.

`N_ENVS` batches `model.act` across N parallel boards each step instead of stepping one board at a time. This is the fix if you're seeing low GPU utilization (a few hundred MB of GPU RAM, single-digit % usage) — a batch of 1 barely exercises the GPU and most of the wall-clock time goes to Python/transfer overhead per step, not compute. Try 16–32 on a Colab GPU; drop to 1 only if you're troubleshooting something and want the simplest possible code path. `ROLLOUT_STEPS` still means *total* env-steps per PPO update (split evenly across `N_ENVS`), so raising `N_ENVS` alone does not change how much data one update sees, only how it's collected.

The curriculum auto-advances to the next stage only once the rolling completion rate over `ADVANCE_WINDOW` episodes clears `ADVANCE_AT` **and** stays there for `ADVANCE_STREAK` consecutive PPO updates in a row — a single lucky rollout no longer promotes the stage (every stage here is a 2+-layer board, so "mostly works" isn't good enough before moving on). `STAGE` below is only the **starting** stage — if a checkpoint already exists for this `RUN_NAME`, its saved stage/step-count/optimizer-state/streak take over and this value is ignored.

In [ ]:
RUN_NAME = "router"          # checkpoint file: {CKPT_DIR}/{RUN_NAME}.pt
STAGE = 0                    # starting curriculum stage (ignored if resuming)
TOTAL_STEPS = 500_000        # stop after this many env steps (cumulative across resumes)
ROLLOUT_STEPS = 2048         # total env steps per PPO update = one "epoch" below
N_ENVS = 16                  # parallel boards per step; raise GPU usage (try 16-32)
ADVANCE_AT = 0.99            # rolling completion rate needed to advance curriculum
ADVANCE_WINDOW = 50          # episodes averaged into the rolling completion rate
ADVANCE_STREAK = 3           # consecutive qualifying updates required to advance
RENDER_EVERY = 5             # show a board snapshot every N epochs
SEED = 0

ckpt_path = Path(CKPT_DIR) / f"{RUN_NAME}.pt"
log_path = Path(CKPT_DIR) / f"{RUN_NAME}_history.jsonl"
torch.manual_seed(SEED)

## Step 3 — Build the model and resume if a checkpoint exists

In [ ]:
model = DualStreamRouter().to(device)
ppo = PPO(model, PPOConfig(), device=device)

stage = STAGE
steps_done = 0
completions = deque(maxlen=max(ADVANCE_WINDOW, N_ENVS))
consecutive_hits = 0
history = []

if ckpt_path.exists():
    ckpt = load_checkpoint(ckpt_path, model, ppo, device=device)
    stage = ckpt.get("stage", STAGE)
    steps_done = ckpt.get("steps_done", 0)
    completions.extend(ckpt.get("completions", []))
    consecutive_hits = ckpt.get("consecutive_hits", 0)
    print(f"resumed '{RUN_NAME}': stage={stage}  steps_done={steps_done:,}  "
          f"rolling completion={np.mean(completions) if completions else 0:.1%}  "
          f"consecutive_hits={consecutive_hits}")
else:
    print(f"no checkpoint at {ckpt_path} -- starting a fresh run at stage {stage}")

# Replay the on-disk history log (if any) into the live plot so the monitor
# picks up right where the last session left off instead of starting blank.
monitor = TrainingMonitor(advance_at=ADVANCE_AT)
if log_path.exists():
    with open(log_path) as f:
        history = [json.loads(line) for line in f]

def make_env(s):
    factory = lambda rng, s=s: generate_board(s, rng)
    if N_ENVS > 1:
        return VecRoutingEnv(factory, N_ENVS, cfg=EnvConfig(), seed=SEED + s)
    return RoutingEnv(factory, cfg=EnvConfig(), seed=SEED + s)

env = make_env(stage)
steps_per_env = max(ROLLOUT_STEPS // N_ENVS, 1) if N_ENVS > 1 else ROLLOUT_STEPS
carried = (None, None)
print(f"stage {stage}: {STAGES[min(stage, len(STAGES)-1)].components}, "
      f"{STAGES[min(stage, len(STAGES)-1)].layers} layers, N_ENVS={N_ENVS}")

## Expected behavior per curriculum stage, and what to do if it doesn't happen

**Measured baseline** (this exact codebase, 49k steps, CPU, stage 0): rolling completion rate went **4.8% → 35.0%**, mean episode return **−115 → −48**, entropy **4.92 → 4.25**, and it was still climbing at cutoff. Everything below stage 0 is an *extrapolation* from that single data point, not a separate measurement — treat the step counts as rough budgets, not guarantees, and use the diagnostics (not the clock) to decide if something is actually wrong.

> **⚠️ Curriculum + board-generation overhaul (July 2026) — start a fresh `RUN_NAME` after pulling it.** Boards are now assembled from a small component footprint library (passives, headers, ICs — `footprints.py`) with realistic pin pitch and courtyard obstacles, instead of floating random pads. The layer cap dropped from 12 to 6, and 4/6-layer boards now have a real stack-up: dedicated power/ground layers that are legal to route through but cost `lam_stackup` if a non-power net dwells there (previously the agent could just hop to an empty layer to dodge congestion). Differential pairs are now real paired topology (matched pitch/width) with a measured skew penalty (`lam4`), not just a random label. A turn-angle penalty (`lam_turn`) and the previous-heading feature push the policy toward straighter routes. The curriculum advance gate is also stricter: `ADVANCE_AT` rose 0.95 → 0.99 and now requires `ADVANCE_STREAK` consecutive qualifying rollouts, not one lucky window. **`NODE_FEAT_DIM`/`HEAD_FEAT_DIM` changed** (8→10, 19→22) — old checkpoints will not load into this model shape at all, so this is a hard break, not just a "don't resume, it's degenerate" situation like the July reward rebalance below.
>
> **⚠️ Reward rebalance (July 2026) — start a fresh `RUN_NAME` after pulling it.** A 174k-step stage-0 run proved the old reward had a degenerate optimum: with shaping weight β = λ₁, moving toward the target earned *net-zero* immediate reward, so the policy converged to "walk to the board edge, aim away from the target, take 0.15mm steps until the budget runs out" (completion collapsed 33% → 3% while return *improved*). Fixes: **β 1.0 → 3.0** (progress now net-positive; see docs/reward-function.md), **commit_snap widened to 2.5/2.0/1.5mm on stages 0–2** (denser completion signal), **value clip rescaled to return units** (`PPOConfig.vf_clip=10`; the old 0.2 froze the critic), and **log-prob/entropy now conditional on action type** (no more crediting unused sub-heads). Consequences: old checkpoints are trained against the old reward and sit in its local optimum — don't resume them; return/entropy magnitudes from the baseline above are **not comparable** to new runs (returns will read higher because approach steps now pay positively). Note β since settled to **1.5** (tightened steering cone) — see `docs/reward-function.md` for the current constants.

| Stage | Board | Rough step budget to reach ~90%+ completion | Notes |
|---|---|---|---|
| 0 | ~3 nets, 2 layers, 20×20mm | 100k–400k | A few passives only, no stack-up. If this stage can't clear ~90% within ~500k steps, the pipeline itself likely has a bug — don't proceed to bigger boards, debug here first. |
| 1 | ~6 nets, 2 layers, 25×25mm | 200k–600k | Adds a header + courtyard keep-outs; congestion-avoidance starts to matter. |
| 2 | ~9 nets, 2 layers, 28×28mm | 300k–800k | Some components mount on the bottom layer (`bottom_mount_prob`) — first stage that *requires* via usage. |
| 3 | ~16 nets, 4 layers, 32×32mm | 500k–1.5M | First stack-up (1 dedicated power/gnd layer) and first differential pair. |
| 4 | ~25 nets, 6 layers, 38×38mm | 1M–3M | 2 dedicated power/gnd layers, more ICs, more diff pairs. |
| 5 | ~27 nets, 6 layers (cap), 45×45mm | 2M+ | Densest board — 6 layers is now the cap (was 12); realistically may need the imitation-warm-start roadmap item (PROJECT.md §8) rather than pure PPO from scratch. |

Net counts above are **approximate** — component placement (decoupling caps, diff pairs) is RNG-dependent now, so the exact count varies board to board; the printed log shows the actual average for whatever finished each rollout.

### Diagnostics — read these off the live plot / printed log each time you check in

- **DRC count must always be 0.** Nothing above depends on it, because it *should* be structurally impossible (actions are legal by construction — see PROJECT.md). If you ever see `info["drc"] > 0` in a rollout, **stop training** and report it — that's a geometry-kernel bug, not a hyperparameter issue.
- **Entropy collapses toward 0 within the first ~10–20 epochs, before completion rises much.** → premature convergence / exploration collapse. `PPOConfig.ent_coef` now anneals automatically (0.02 → 0.003 over `ent_anneal_steps`, default 500k) via `ppo.update(buf, steps_done)`, so this should be rarer than before; if it still happens early, raise `PPOConfig.ent_coef` or `ent_anneal_steps`.
- **Entropy stays high/flat *and* completion rate is flat *and* `pi_loss` hovers near 0 for many epochs, but `commit_rate` is also low.** → the agent rarely reaches the window where COMMIT becomes legal at all -- it's not finding targets. Diagnose *what the policy actually does* before touching hyperparameters: run rollouts and measure mean step length (mm), aim (cos between chosen angle and target direction), and closest approach per net vs `commit_snap`. Step length collapsing toward `min_segment_length` + aim cos ≤ 0 means it's minimizing movement penalties instead of routing — a reward-structure problem, not an exploration problem.
- **Return/completion rises for a while, then crashes and stays down, with entropy and `v_loss` spiking together right at the crash.** → a destructively large PPO update knocked the policy out of a good region (the critic's sudden bad predictions, visible as a `v_loss` spike, and the policy briefly getting more random again, visible as an `entropy` spike, tend to show up together at the same step). Check `clip_frac` in the printed log around that point -- above ~30% confirms updates are too large. This pattern is why `PPOConfig.lr` was lowered (3e-4 → 1e-4) and PPO2-style value clipping exists (`PPOConfig.vf_clip`, in *return* units — it was briefly 0.2, which froze the critic; keep it on the scale of one net completion, ~10). If it still happens, lower `lr` further or reduce `PPOConfig.epochs` from 4 to 2-3.
- **Completion rate genuinely plateaus** (flat for 100k+ steps, not still trending up, and not the crash-then-stuck pattern above) well below the stage's rough budget. → this is the expected failure mode for pure RL-from-scratch on harder boards; see PROJECT.md §8 item 2 (A\* teacher + imitation warm-start) before spending more compute on more PPO steps.
- **Stage seems stuck just below `ADVANCE_AT` for a long time.** → this is partly by design now (0.99 + a 3-update streak is intentionally stricter than the old single-window 0.95), not necessarily a bug. Check whether the rolling completion rate is genuinely plateaued (see above) vs. just noisy around the threshold — the streak counter resets on any qualifying-window miss, so a policy oscillating around ~0.97-0.99 can take a while to string together 3 in a row.
- **steps/s drops noticeably over a long session.** → usually Drive I/O latency from saving a checkpoint every single epoch, not a code leak. If it's a problem, raise the checkpoint cadence (edit the loop below to save every 3rd epoch instead of every epoch).
- **GPU usage stays low (a few hundred MB VRAM, low utilization).** → raise `N_ENVS` (Step 2) so `model.act` sees a real batch instead of one board at a time.
- **Something is unrecoverably wrong and you want to restart this stage from scratch.** → delete `{RUN_NAME}.pt` and `{RUN_NAME}_history.jsonl` from `CKPT_DIR` (or just change `RUN_NAME`) and re-run from Step 3.

## Step 4 — Train

One "epoch" here = one PPO rollout (`ROLLOUT_STEPS` env steps) + one PPO update. Each epoch: the plot redraws, the checkpoint + history log save to Drive, and every `RENDER_EVERY` epochs a board snapshot from the current policy is shown inline.

**Safe to interrupt at any time** (Colab "Interrupt execution", or just closing the tab): the last completed epoch's checkpoint is already on Drive. Re-running this cell continues from there — it does not restart the epoch that was interrupted mid-flight, only from the last one that finished.

In [ ]:
try:
    while steps_done < TOTAL_STEPS:
        t0 = time.time()
        if N_ENVS > 1:
            buf, stats, carried = collect_rollout_vec(env, model, steps_per_env, device, *carried, ppo.cfg)
            steps_this_update = steps_per_env * N_ENVS
        else:
            buf, stats, carried = collect_rollout(env, model, ROLLOUT_STEPS, device, *carried, ppo.cfg)
            steps_this_update = ROLLOUT_STEPS
        upd = ppo.update(buf, steps_done)
        steps_done += steps_this_update
        completions.extend(stats["completions"])

        drc_total = int(sum(stats["drc"]))
        # Net count is RNG-dependent now (component placement can fail, the
        # general pool trims an odd leftover pad) -- average whatever
        # finished this rollout instead of reading a static per-stage count.
        mean_nets_total = float(np.mean(stats["nets_total"])) if stats["nets_total"] else float("nan")
        mean_cmp = float(np.mean(completions)) if completions else 0.0
        record = {
            "steps_done": steps_done, "stage": stage,
            "ep_return": float(np.mean(stats["returns"])) if stats["returns"] else float("nan"),
            "completion": mean_cmp,
            "nets_total": mean_nets_total,
            "drc": drc_total, "commit_rate": stats["commit_rate"],
            "steps_per_sec": steps_this_update / (time.time() - t0),
            **upd,
        }
        history.append(record)
        with open(log_path, "a") as f:
            f.write(json.dumps(record) + "\n")

        # Consistency gate: the window must be full AND clear ADVANCE_AT,
        # AND that has to hold for ADVANCE_STREAK updates in a row -- a
        # single lucky window no longer promotes the stage.
        qualifies = len(completions) == completions.maxlen and mean_cmp >= ADVANCE_AT
        consecutive_hits = consecutive_hits + 1 if qualifies else 0
        save_checkpoint(ckpt_path, model, ppo, stage, steps_done, completions, history,
                        consecutive_hits=consecutive_hits)

        if drc_total > 0:
            print(f"  !! DRC={drc_total} this rollout -- geometry-kernel bug, not a "
                  f"hyperparameter issue. Stop and report this. See CLAUDE.md invariant #2.")
        if upd["clip_frac"] > 0.3:
            print(f"  ! clip_frac={upd['clip_frac']:.1%} is high -- updates may be too "
                  f"large/destabilizing. If ep_return/completion crash and stay down after "
                  f"a run of high clip_frac, that's why (see PPOConfig.lr / value clipping).")

        epoch = len(history)
        if epoch % RENDER_EVERY == 0:
            demo_env = RoutingEnv(lambda rng, s=stage: generate_board(s, rng), cfg=EnvConfig(),
                                  seed=int(np.random.default_rng().integers(1_000_000)))
            d_obs, d_masks = demo_env.reset()
            done = False
            with torch.no_grad():
                while not done:
                    t_masks = {k: torch.from_numpy(v).unsqueeze(0).to(device) for k, v in d_masks.items()}
                    a, _, _ = model.act(to_torch(d_obs, device), t_masks, deterministic=True)
                    d_obs, d_masks, _, done, d_info = demo_env.step(
                        (int(a.action_type), int(a.angle_bin), float(a.dist_frac), int(a.layer)))
            show_board_inline(demo_env.board,
                              title=(f"epoch {epoch} | steps {steps_done:,} | stage {stage} | "
                                    f"{d_info['nets_done']}/{d_info['nets_total']} nets | DRC={d_info['drc']}"),
                              completed=demo_env.completed)

        monitor.update(record)
        mean_nets_routed = mean_cmp * mean_nets_total
        print(f"steps {steps_done:>8}  stage {stage}  ep_return {record['ep_return']:8.2f}  "
              f"completion {record['completion']:5.1%} ({mean_nets_routed:.2f}/{mean_nets_total:.1f} nets)  "
              f"entropy {upd['entropy']:6.3f}  pi {upd['pi_loss']:+.4f}  v {upd['v_loss']:8.3f}  "
              f"clip {upd['clip_frac']:5.1%}  drc {drc_total}  commit_rate {stats['commit_rate']:5.1%}  "
              f"{record['steps_per_sec']:6.0f} steps/s")

        if consecutive_hits >= ADVANCE_STREAK and stage < len(STAGES) - 1:
            stage += 1
            print(f"=== curriculum advance -> stage {stage} "
                  f"(after {ADVANCE_STREAK} consecutive qualifying updates) ===")
            env = make_env(stage)
            completions.clear()
            consecutive_hits = 0
            carried = (None, None)
            save_checkpoint(ckpt_path, model, ppo, stage, steps_done, completions, history,
                            consecutive_hits=consecutive_hits)

    print("training complete")
except KeyboardInterrupt:
    print(f"interrupted at step {steps_done:,} -- last checkpoint on Drive is up to date, safe to stop.")

## Resuming after a disconnect

Colab disconnects, or you close the tab, or the runtime recycles. Nothing is lost as long as Drive had synced (it saves every epoch). To pick back up:

1. Reopen this notebook (from Drive, or re-clone/open from GitHub).
2. Run **Step 1** (mount Drive + `git pull` — picks up any code changes pushed to GitHub since your last session).
3. Run **Step 2** with the *same* `RUN_NAME` you used before.
4. Run **Step 3** — it will print `resumed '...'` with the stage/step count it found.
5. Run **Step 4** — training continues exactly where it left off, optimizer momentum and all.

You do **not** need to re-run Step 4 with different `TOTAL_STEPS` math — it's a cumulative target across the whole run, not per-session.

## Extra: render one policy rollout on demand

In [ ]:
demo_env = RoutingEnv(lambda rng, s=stage: generate_board(s, rng), cfg=EnvConfig(), seed=123)
d_obs, d_masks = demo_env.reset()
done = False
with torch.no_grad():
    while not done:
        t_masks = {k: torch.from_numpy(v).unsqueeze(0).to(device) for k, v in d_masks.items()}
        a, _, _ = model.act(to_torch(d_obs, device), t_masks, deterministic=True)
        d_obs, d_masks, _, done, d_info = demo_env.step(
            (int(a.action_type), int(a.angle_bin), float(a.dist_frac), int(a.layer)))
show_board_inline(demo_env.board,
                  title=f"stage {stage} | {d_info['nets_done']}/{d_info['nets_total']} nets | DRC={d_info['drc']}",
                  completed=demo_env.completed)